In [1]:
import torch

# 自动微分模块 = 对损失函数求导，结合反向传播，更新权重参数w，b（重要）
训练神经网络时,最常用的算法就是  反向传播  。在该算法中,参数(模型权重也就是图1中的w和b)会根据 损失函数关于对应参数的梯度 进行调整。为了计算这些梯度,PyTorch内置了名为 torch. autograd 的微分模块。它支持任意计算图的自动梯度计算:

                                            图1

                torch. autograd 的微分模块 --> 导数 --> 更新w，b --> 优化训练

图中X是特征，w是权重，b是偏置 --> 多元线性公式：y=wx+b

--> 权重（w）更新公式：W_新 = W_旧 - 学习率 * 损失函数的导数。其中损失函数的导数就是计算的梯度。
--> 偏置（b）更新公式：b_新 = b_旧 - 学习率 * 损失函数的导数。其中损失函数的导数就是计算的梯度。
实际开发中，大多数情况下，可以不考虑偏置，会将偏置矩阵直接设置为全0矩阵

图中 * 表示乘法，+ 表示加法，z 表示预测值， y 表示真实值。MSE 图中表示z与y的均方误差（所有误差的平方和除以样本数），均方根误差对MSE 开平方根。loss是损失函数，对其求导（自动微分模块），得到的结果就是梯度。将梯度带入权重（w）更新公式中，算出新的w,b。继续图1过程。

将梯度带入权重（w）更新公式中，算出新的w,b，参与下一过程就是反向传播。
由 X 到 Z （包括w,b）这一过程称为正向传播，正向传播得到loss

接下来我们使用这个结构进行自动微分模块的介绍。我们使用 backward 方法、grad 属性来实 现梯度的计算和访问
# 梯度计算
pytorch不支持向量张量求导对向量张量求导，只支持标量张量对向量张量的求导。
其中，标量张量是向量张量的一个值或参数，例如向量张量：Tensor([a,b,c])那么标量张量Tensor(a)，相当于求偏导

# 关于 Autograd 的几个概念
backward函数  计算梯度返回的是向量，所以在多次求导时，应加上一个sum函数，加上每个参数的偏导成为全导，将向量转成标量
例：y(标量):y.sum().backward()
> torch.autograd.backward(tensors,grad_tensors=None,retain_graph=None, create_graph=False)
> 重点：tensor: 用于计算梯度的tensor, 两种调用方式：torch.autograd.backward(z) == z.backward()
> 重点：grad_tensors: 在计算矩阵的梯度时会用到。他其实也是一个tensor,shape 一般需要和前面的tensor保持一致。
> 重点：retain_graph: 通常在调用一次backward后,pytorch会自动把计算图销毁,所以要想对某个变量重复调用backward,则需要将该参数设置为True
> 重点：create_graph: 如果为True,那么就创建一个专门的graph of the derivative记录本次求导的结果,这可以方便迭代计算高阶微分。

计算x点的梯度值：x.grad

梯度：是一个向量，函数值变化最快的方向
梯度值：是一个标量、导数，函数值沿梯度变化值的大小

In [4]:
"""
案例:
    演示自动微分模块,具体如何求导.
    权重更新公式: W新 = W旧 - 学习率 * 梯度
    梯度 = 损失函数的导数
"""
#一轮计算：
# 1. 定义变量,记录:初始的权重w(旧)
w = torch.tensor(10,requires_grad=True,dtype=torch.float) # 只有标量才能求导，且大多数底层操作都是 浮点型，需要转型
# 2. 定义loss变量,表示损失函数
loss = 2*w**2 # loss在实际运用过程中，不是人为给定
# 3. 打印梯度函数类型(了解)
print(f"梯度函数类型：{type(loss.grad_fn)}")
# 4. 计算梯度,梯度 = 损失函数的导数,计算完毕后,会记录到 w.grad
loss.sum().backward() #会记录到 w.grad
# 5. 代入 权重更新公式: W新 = WH - 学习率 * 梯度
W_new = w.data - 0.01*w.grad # 以后多次计算权重时 W_new 应该换为w.data。迭代更新
                             # 学习率0.01 在以后也不是人为给定
print(f"原来的权重{w.data}，现在的权重{W_new}")

梯度函数类型：<class 'MulBackward0'>
原来的权重10.0，现在的权重9.600000381469727


In [9]:
"""
梯度下降求法求最优解
案例:
演示自动微分模块,循环实现 计算梯度,更新参数.
需求:
求 loss= w**2 + 20 的极小值点 并打印loss是最小值时 w的值(梯度)

解题步骤:
1. w=10 requires_grad=True dtype=torch. float32
w=10的解释：在梯度下降算法中，您需要一个初始的"猜测值"开始。由于函数 w**2 + 20 的最小值在 w=0 处，从 w=10 开始提供了一个清晰的例子，说明算法如何迭代地向最优解移动。选择 10 是任意的但便于演示——它距离最优解足够远，可以清楚地显示优化过程。
2. 定义函数 loss = w**2 + 20
3. 利用梯度下降法 循环迭代1000 求最优解
3.1 正向计算(前向传播)
3.2 梯度清零 w.grad.zero_()
3.3 反向传播
3.4 梯度更新 w.data = w.data - 0.01 * w.grad
"""
# 1. w=10 requires_grad=True dtype=torch. float32
w = torch.tensor(10,requires_grad=True,dtype=torch.float32)
# 2. 定义函数 loss = w**2 + 20
loss = w ** 2 + 20
# 3. 利用梯度下降法 循环迭代100 求最优解
print(f"开始 权重{w.data},(0.01 * w.grad):无,loss:{loss}")
for i in range(1,101):
    # 3.1 正向计算(前向传播)
    loss = w ** 2 + 20
    # 3.2 梯度清零 w.grad.zero_()
    if w.grad is not None:
        w.grad.zero_()
    # 3.3 反向传播
    loss.sum().backward()
    # 3.4 梯度更新 w.data = w.data - 0.01 * w.grad
    w.data = w.data - 0.01 * w.grad
    print(f"第{i}次，权重初始值：{w.data:.5f},(0.01 * w.grad):{0.01 * w.grad:.5f},loss:{loss:.5f}")

开始 权重10.0,(0.01 * w.grad):无,loss:120.0
第1次，权重初始值：9.80000,(0.01 * w.grad):0.20000,loss:120.00000
第2次，权重初始值：9.60400,(0.01 * w.grad):0.19600,loss:116.04000
第3次，权重初始值：9.41192,(0.01 * w.grad):0.19208,loss:112.23682
第4次，权重初始值：9.22368,(0.01 * w.grad):0.18824,loss:108.58425
第5次，权重初始值：9.03921,(0.01 * w.grad):0.18447,loss:105.07632
第6次，权重初始值：8.85842,(0.01 * w.grad):0.18078,loss:101.70729
第7次，权重初始值：8.68126,(0.01 * w.grad):0.17717,loss:98.47168
第8次，权重初始值：8.50763,(0.01 * w.grad):0.17363,loss:95.36420
第9次，权重初始值：8.33748,(0.01 * w.grad):0.17015,loss:92.37978
第10次，权重初始值：8.17073,(0.01 * w.grad):0.16675,loss:89.51353
第11次，权重初始值：8.00731,(0.01 * w.grad):0.16341,loss:86.76079
第12次，权重初始值：7.84717,(0.01 * w.grad):0.16015,loss:84.11706
第13次，权重初始值：7.69022,(0.01 * w.grad):0.15694,loss:81.57802
第14次，权重初始值：7.53642,(0.01 * w.grad):0.15380,loss:79.13953
第15次，权重初始值：7.38569,(0.01 * w.grad):0.15073,loss:76.79761
第16次，权重初始值：7.23798,(0.01 * w.grad):0.14771,loss:74.54843
第17次，权重初始值：7.09322,(0.01 * w.grad):0.14476,loss:72.3

In [13]:
"""
案例：
    演示 detach()函数的功能,解决 自动微分的弊端.
回顾:
    自动微分 =求导,即:基于损失函数,计算梯度. 结合权重更新公式:w新 = W旧 - 学习率 * 梯度,来更新权重的.
问题:
    一个张量一旦设置了自动微分,这个张量就不能直接转成 numpy的 ndarray对象了,需要通过 detach()函数解决
"""
import numpy as np
# 1. 定义张量.
t1 = torch.tensor([10,20],requires_grad=True,dtype=torch.float)
print("t1",t1,"type(t1)",t1,type(t1))
print("-"*30)
# 2. 尝试把上述的张量 → numpy对象.
t2 = t1.detach()
n1 = t2.numpy()
print("n1",n1,"type(n1)",type(n1))
print("-"*30)
# 3. 测试上述的t1 和 t2是否共享同一块空间 → 共享.
t1.data[0] = 30
print("t1",t1,"type(t1)",type(t1))
print("t2",t2,"type(t2)",type(t2))
print("-"*30)
# 4. 查看t1 和 t2谁可以自动微分.
print("t1.requires_grad",t1.requires_grad)
print("t2.requires_grad",t2.requires_grad)

t1 tensor([10., 20.], requires_grad=True) type(t1) tensor([10., 20.], requires_grad=True) <class 'torch.Tensor'>
------------------------------
n1 [10. 20.] type(n1) <class 'numpy.ndarray'>
------------------------------
t1 tensor([30., 20.], requires_grad=True) type(t1) <class 'torch.Tensor'>
t2 tensor([30., 20.]) type(t2) <class 'torch.Tensor'>
------------------------------
t1.requires_grad True
t2.requires_grad False


In [ ]:
"""
案例:
    演示自动微分的真实应用场景.
结论:
    1. 先前向转播(正向传播) 计算出 预测值(z)
    2. 基于损失函数,结合 预测值(z) 和 真实值(y),来计算 梯度.
    3. 结合权重更新公式 w新 = W旧 - 学习率 * 梯度,来更新 权重.
"""

torch.autograd.grad()
> def grad(outputs, inputs, grad_outputs=None, retain_graph=None, create_graph=False, only_inputs=True, allow_unused=False)

计算和返回output关于inputs的梯度的和，也就是可以求解多个Tensor的和。
重点：outputs : 函数的因变量,即需要求导的那个函数
重点：inputs : 函数的自变量,可以定义多个Tensor，也就是可以求解多个Tensor的和。
重点：grad_outputs同backward函数的grad_tensor: 在计算矩阵的梯度时会用到。他其实也是一个tensor,shape 一般需要和前面的tensor保持一致
重点：retain_graph: 通常在调用一次函数后,pytorch会自动把计算图销毁,所以要想对某个变量重复调用backward,则需要将该参数设置为True
重点：create_graph: 如果为True,那么就创建一个专门的graph of the derivative记录本次求导的结果,这可以方便迭代计算高阶微分。
only_inputs: 只计算input的梯度
allow_unused( bool, 可选):如果为False,当计算输出出错时(因此他们 的梯度永远是0)指明不使用的inputs。


torch.autograd.Function
> 每一个原始的自动求导运算实际上是两个在Tensor上运行的函数

一个函数是forward函数计算从输入Tensors获得的输出Tensors
一个函数是backward函数接收输出Tensors对于某个标量值的梯度,并且计算输 入Tensors相对于该相同标量值的梯度
最后,利用apply方法执行相应的运算
● 定义在Function类的父类_FunctionBase中定义的一个方法